# Day 4: Session 4C - The Derived-Column Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/4c_transforming_data.html)

Date: 09/03/2026

### write the derived-column pattern, 
### **df['new'] = expression**


In [48]:
import pandas as pd
import numpy as np

url = 'https://eds-217-essential-python.github.io/data/marine_microplastics.csv'
plastics = pd.read_csv(url)

m3 = plastics[plastics['Unit'] == 'pieces/m3'].copy()
m3.shape

(10178, 22)

In [49]:
# derived-column pattern
# df['new_column'] = df['existing_column'] */+ operations 

m3['pieces_per_liter'] = m3['Measurement'] / 1000

In [50]:
m3[['Measurement', 'Unit', 'pieces_per_liter']].head()

,Measurement,Unit,pieces_per_liter
0,0.020000,pieces/m3,0.000020
1,0.008000,pieces/m3,0.000008
2,0.019886,pieces/m3,0.000020
3,0.018000,pieces/m3,0.000018
4,0.000000,pieces/m3,0.000000


In [51]:
m3.shape

(10178, 23)

In [52]:
m3['latitude_radians'] = m3['Latitude'] * (3.141592653589793 / 180)
m3[['latitude_radians', 'Latitude']].head()

,latitude_radians,Latitude
0,-1.019766,-58.428300
1,-0.895497,-51.308200
2,-0.904546,-51.826667
3,-0.553200,-31.696000
4,0.110828,6.350000


In [53]:
url = 'https://eds-217-essential-python.github.io/data/banana_index.csv'
foods = pd.read_csv(url, index_col='entity')
foods[['emissions_kg', 'land_use_kg', 'Bananas index (kg)']].head()

,emissions_kg,land_use_kg,Bananas index (kg)
entity,,,
Ale,0.488690,0.811485,0.559558
Almond butter,0.387011,7.683045,0.443134
Almond milk,0.655888,1.370106,0.751002
Almonds,0.602368,8.230927,0.689721
Apple juice,0.458378,0.660629,0.524851



build a new column from a column and a number, and from two columns


In [54]:
banana_emissions = foods.loc['Bananas', 'emissions_kg']
banana_emissions

0.87334957

In [55]:
foods['my_banana_index'] = foods['emissions_kg'] / banana_emissions
foods[['emissions_kg', 'Bananas index (kg)', 'my_banana_index']].head()


,emissions_kg,Bananas index (kg),my_banana_index
entity,,,
Ale,0.488690,0.559558,0.559558
Almond butter,0.387011,0.443134,0.443134
Almond milk,0.655888,0.751002,0.751002
Almonds,0.602368,0.689721,0.689721
Apple juice,0.458378,0.524851,0.524851


In [56]:
foods['land_per_emission'] = foods['land_use_kg'] / foods['emissions_kg']
foods['land_per_emission'].sort_values(ascending=False).head(5)

entity
Almond butter    19.852250
Almonds          13.664286
Beans            12.428466
Chickpeas        11.578514
Lentils          10.831714
Name: land_per_emission, dtype: float64


### use np.log10() when a column spans several orders of magnitude


In [57]:
m3['Measurement'].describe()

count     10178.000000
mean        219.409152
std        2599.554575
min           0.000000
25%           0.000000
50%           0.007200
75%           0.049937
max      110480.000000
Name: Measurement, dtype: float64

In [58]:
positive = m3[m3['Measurement'] > 0].copy()
positive.shape
# filtering out zeros

(7091, 24)

In [59]:
positive['log10_measurement'] = np.log10(positive['Measurement'])
positive[['Measurement', 'log10_measurement']].head()

,Measurement,log10_measurement
0,0.020000,-1.698970
1,0.008000,-2.096910
2,0.019886,-1.701453
3,0.018000,-1.744727
5,0.013000,-1.886057


In [60]:
positive['log10_measurement'].describe()

count    7091.000000
mean       -1.254441
std         1.502175
min        -3.170053
25%        -2.188425
50%        -1.665546
75%        -0.879686
max         5.043284
Name: log10_measurement, dtype: float64

In [61]:
positive['log10_per_liter'] = np.log10(positive['pieces_per_liter'])
positive[['pieces_per_liter', 'log10_per_liter']]

,pieces_per_liter,log10_per_liter
0,0.000020,-4.698970
1,0.000008,-5.096910
2,0.000020,-4.701453
3,0.000018,-4.744727
5,0.000013,-4.886057
...,...,...
16238,0.000006,-5.221849
16239,0.000004,-5.397940
16240,0.000326,-3.486782
16241,0.000011,-4.958607



tidy a text column with .str.strip(), .str.lower(), and .str.replace()

In [62]:
# .str removes any whitespace
# I.e. 
plastics[plastics['Keywords'] == 'R/V Tara'].shape
# shows no rows with R/V Tara

(0, 22)

In [63]:
plastics['Keywords'].unique()[10:14]
# Wait but there's R/V Tara!

array(['Amazon Continental Shelf', 'Antarctic Circumnavigation Expedition',
       'R/V Tara ', 'SV Mir; ORV Alguita; SV Sea Dragon; RV Stad Amsterdam'],
      dtype=object)

In [64]:
# Need to remove space from 'R/V Tara ' with .str
plastics['Keywords'] = plastics['Keywords'].str.strip()
plastics[plastics['Keywords'] == 'R/V Tara'].shape

(23, 22)

Strip text columns as a reflex, the way you check .isnull().sum() as a reflex. Trailing spaces are invisible, they survive every copy and export, and they break exact matches silently

In [65]:
plastics['Density Class'].value_counts()

Density Class
Medium       8029
Very Low     4155
Low          1944
High         1671
Very High     446
Name: count, dtype: int64

In [66]:
# str.lower() makes matching pedictable
plastics['density_class'] = plastics['Density Class'].str.lower()
plastics['density_class'].value_counts()

density_class
medium       8029
very low     4155
low          1944
high         1671
very high     446
Name: count, dtype: int64

In [67]:
# .str.replace() swaps one piece of text for another
# I.e. remove "Ocean" from "Atlantic", "Pacific", etc
plastics['ocean'] = plastics['Oceans'].str.replace(' Ocean', '') #Replace Ocean_ with nothing
plastics['ocean']


0        Atlantic
1        Atlantic
2         Pacific
3        Atlantic
4         Pacific
           ...   
16240    Atlantic
16241    Atlantic
16242    Atlantic
16243    Atlantic
16244    Atlantic
Name: ocean, Length: 16245, dtype: object

## Putting it all together !

In [68]:
url = 'https://eds-217-essential-python.github.io/data/marine_microplastics.csv'
plastics = pd.read_csv(url)

# Clean: drop the rows with no measurement (session 4A), then strip the stray
# whitespace out of Keywords (this session)
plastics = plastics.dropna(subset=['Measurement'])
plastics['Keywords'] = plastics['Keywords'].str.strip()

# Select one unit, so arithmetic means something (session 3B)
samples = plastics[plastics['Unit'] == 'pieces/m3'].copy()
samples = samples[samples['Measurement'] > 0].copy()

# Transform (this session)
samples['ocean'] = samples['Oceans'].str.replace(' Ocean', '')
samples['pieces_per_liter'] = samples['Measurement'] / 1000
samples['log10_measurement'] = np.log10(samples['Measurement'])

samples[['ocean', 'Measurement', 'pieces_per_liter', 'log10_measurement']].head()

,ocean,Measurement,pieces_per_liter,log10_measurement
0,Atlantic,0.020000,0.000020,-1.698970
1,Atlantic,0.008000,0.000008,-2.096910
2,Pacific,0.019886,0.000020,-1.701453
3,Atlantic,0.018000,0.000018,-1.744727
5,Pacific,0.013000,0.000013,-1.886057


### Key points
The derived-column pattern is df['new'] = expression. 
Assignment creates the column, and unlike most pandas operations it changes the table in place.

Arithmetic on a column applies to every row at once. No loop.

Naming two columns on the right side computes row by row, matched up by row label.

Reusing an existing name overwrites it with no warning.

np.log10() rescales a column that spans orders of magnitude. Filter out zeros first, because the logarithm of zero is undefined.

Text columns are cleaned through .str: .strip() for invisible whitespace, .lower() for predictable matching, .replace(old, new) for everything else. One method per line.